# Notebook 02 — A Knowledge Base for an AI SQL Assistant

**VidyaSankalp · Applied GenAI Engineering** · Tutor: Prudhvi Akella

This is the heart of the course. In Part 0 we stored five help articles and found them by meaning. Now we build
what a real AI assistant needs: a **knowledge base** that lets an LLM write correct SQL for a database it has never
seen.

```text
1  THE QUESTION        "What was our GMV last month?"  → the assistant must write SQL
2  WHAT THE LLM        which table? which column? what does GMV mean HERE? is there data?
   DOESN'T KNOW
3  WHY NOT PASTE       the whole knowledge ≈ 42,000 tokens — for every question
   EVERYTHING?            └─► store it, retrieve only the relevant pieces
4  THE KNOWLEDGE BASE  6 tabs, each answers one thing the LLM needs
5  THE DESIGN  ★       one row → one record: what to embed · what id · what metadata · what to skip
6  BUILD               337 records
7  LOAD                into Pinecone, namespace ecom-kb
8  RETRIEVE            the agent's three lookups: schema · examples · common
9  THE CONTEXT PACK ★  retrieved + followed + always in — what the LLM receives
10 KEEP IN SYNC        the workbook changes → the index follows
11 WHAT IT CAN'T DO    meaning, not structure
```

**Needs:** Part 0 done (same index `ecom-kb-tutorial`), the same `.env` (OpenAI + Pinecone keys), and
`kb_feed_ecom.xlsx` in this folder. **Time:** about 2½ hours.

## Chapter 1 — The question

An online store has a database with 14 tables: orders, order items, payments, shipments, customers, products,
suppliers and more. A manager types into an AI assistant:

> **"What was our GMV last month?"**

The assistant is an LLM. It must turn the question into SQL, run it, and answer. To do that it needs to know:

```text
"What was our GMV last month?"
        │
        ▼
   ┌─────────┐      SELECT ??? FROM ??? WHERE ???      ┌──────────┐
   │   LLM   │ ──────────────────────────────────────► │ database │ ──► answer
   └─────────┘                                          └──────────┘
        which table?  which column?  what is GMV?  which dates exist?
```

## Chapter 2 — What the LLM doesn't know

An LLM knows SQL and it knows what GMV usually means. It does **not** know **this** company's database. Left alone,
it guesses something like:

```sql
SELECT SUM(gmv) FROM sales WHERE order_date >= DATE_TRUNC('month', CURRENT_DATE - INTERVAL '1 month');
```

Wrong in four ways — and each fix is a fact that lives only in this company's documentation:

| The guess | The truth (from `kb_feed_ecom.xlsx`) | Where that fact lives |
|---|---|---|
| table `sales` | there is no `sales` table; orders are in `ecom.orders` | tab `nlq_table` |
| column `gmv` | no column is called GMV; it is `orders.total_amount` | tab `nlq_column` |
| GMV = everything paid | here GMV = **Revenue** = merchandise only — shipping, tax and discount **not** included | tab `glossary` |
| "last month" = the month before today | orders run **2023-01-01 to 2024-12-28**; nothing later exists | tab `capability_card` |

Plus two things that make the answer *good*, not just valid:

| Helpful fact | Where it lives |
|---|---|
| A proven query that already answers "revenue by month" | tab `nlq_examples` |
| How tables connect, when a question needs more than one | tab `nlq_join` |

> **So the LLM needs the right facts with every question.** Why not just give it all the facts, every time?

## Chapter 3 — Why not paste everything into the prompt?

Let's measure. First, setup — keys in `.env`, never in the notebook.

In [10]:
%pip install -q openai pinecone python-dotenv openpyxl


[notice] A new release of pip is available: 24.3.1 -> 26.3
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [11]:
# Setup for this part: connect to OpenAI and Pinecone.
# No Excel data is loaded yet.

import os
import time  # Can be used later for delays or retries.

import openpyxl  # Will be used later to read Excel data.
from dotenv import load_dotenv
from openai import OpenAI
from pinecone import Pinecone, ServerlessSpec

# Load API keys from the .env file.
load_dotenv()

# Check that both API keys are available before connecting.
for key in ("OPENAI_API_KEY", "PINECONE_API_KEY"):
    if not os.environ.get(key):
        raise ValueError(f"Please set {key} in your .env file.")

# Create the OpenAI client.
client = OpenAI()

# Use this same embedding model for both stored records and search questions.
# It creates vectors with 1536 numbers.
EMBED_MODEL = "text-embedding-3-small"

# Use the same index as Part 0 and store this part's data in its own namespace.
INDEX_NAME = "ecom-kb-tutorial"
NAMESPACE = "ecom-kb"


def embed(texts):
    """Turn a list of texts into a list of embedding vectors."""
    # Send all the texts to OpenAI in one request.
    response = client.embeddings.create(
        model=EMBED_MODEL,
        input=texts,
    )

    # Return one vector for each text, in the same order.
    return [item.embedding for item in response.data]


# Connect to Pinecone using the API key.
pc = Pinecone(api_key=os.getenv("PINECONE_API_KEY"))

# Create the index only if it doesn't already exist.
# If your Pinecone project has reached its index limit, use an existing
# compatible index instead of creating another one.
if not pc.has_index(INDEX_NAME):
    pc.create_index(
        name=INDEX_NAME,
        dimension=1536,
        metric="cosine",
        spec=ServerlessSpec(cloud="aws", region="us-east-1"),
    )

# Get the index object so later code can store and search vectors.
index = pc.Index(INDEX_NAME)

print("Connected to", INDEX_NAME)

Connected to ecom-kb-tutorial


In [12]:
# Open the Excel workbook in read-only mode.
# Read-only mode is suitable here because we only inspect the file.
# This code counts rows and characters; it does not create embeddings.
workbook = openpyxl.load_workbook(
    "./docs/kb_feed_ecom.xlsx",
    read_only=True,
)

# List the worksheet tabs we want to measure.
KB_TABS = [
    "nlq_table",
    "nlq_column",
    "nlq_join",
    "glossary",
    "nlq_examples",
    "capability_card",
]


def read_sheet(sheet_name):
    """Return the non-empty data rows from a worksheet as dictionaries."""
    # Select one worksheet by its tab name.
    worksheet = workbook[sheet_name]

    # Read cell values as rows of Python values.
    rows = worksheet.iter_rows(values_only=True)

    # The first row is assumed to contain column names, such as "table_name".
    column_names = next(rows)

    # This list will hold the worksheet's data rows.
    records = []

    # Go through the remaining rows, one at a time.
    for values in rows:
        # Ignore rows where every cell is empty.
        if all(value is None for value in values):
            continue

        # Pair each column name with its value from this row.
        # For example: {"table_name": "orders", "description": "Customer orders"}
        record = {}
        for column_name, value in zip(column_names, values):
            record[column_name] = value

        # Save this row so the caller can count its text later.
        records.append(record)

    return records


# This will hold the character count across all selected worksheets.
total_characters = 0

# Measure each worksheet separately.
for tab_name in KB_TABS:
    # Read the worksheet's non-empty rows.
    records = read_sheet(tab_name)
    print(records[0])
    print("--------------------")

{'action': 'INSERT', 'effective_date': '2026-10-02', 'change_reason': 'initial load — e-commerce NLQ collection', 'table': 'ecom.orders', 'description': "# Orders\n**TABLE:** ecom.orders\n**BUSINESS DOMAIN:** E-commerce / Online Retail (US)\n\n## BUSINESS DESCRIPTION\nEvery order placed on the online store between 2023-01-01 and 2024-12-28: who ordered, when, how they paid, where it ships, how it was fulfilled, and the order's status. The header of an order — the products bought are in order_items.\n\n## ANALYTICAL GRAIN\nOne row per order. order_id is unique.\n\n## KEY BUSINESS ENTITIES\nOrder; customer; shipping destination; payment method; coupon; referral source; device.\n\n## KEY MEASURES AND ATTRIBUTES\ntotal_amount (merchandise subtotal — the amount paid); shipping_cost, tax_amount and discount_amount are stored beside it but NOT included in total_amount. Dimensions: order_status, payment_method, shipping_method, device_type, referral_source, coupon_code, is_gift, shipping_state

## Chapter 4 — The knowledge base: six tabs

`kb_feed_ecom.xlsx` describes the database for the assistant. Each tab answers one thing the LLM needs to know:

| Tab | Rows | Answers the LLM's question | Example row |
|---|---|---|---|
| `nlq_table` | 14 | **What tables exist, and what is in each?** | `ecom.orders` — every order: who, when, how paid, status |
| `nlq_column` | 185 | **Which column holds this? What do people call it?** | `orders.total_amount` — merchandise subtotal; also called Revenue, Sales |
| `nlq_join` | 16 | **How do two tables connect?** | `orders` → `customers` on `customer_id`, many-to-one |
| `glossary` | 66 | **What does this business word mean here?** | **Revenue** = SUM(orders.total_amount); also called sales, GMV |
| `nlq_examples` | 61 | **Is there a proven query I can copy?** | "Which brands generate the most revenue?" + its SQL |
| `capability_card` | 3 | **What can — and can't — I answer?** | "orders after 2024-12-28 … nothing later" |

The workbook also has an `Instructions` tab — written for people who maintain the workbook, not for the assistant.
We don't load it.

```text
"What was our GMV last month?"   needs facts from:
   glossary         GMV → Revenue → merchandise only → orders.total_amount
   nlq_column       ecom.orders.total_amount (FLOAT)
   nlq_table        ecom.orders
   nlq_examples     "What is total merchandise revenue by month?" + SQL
   capability_card  data ends 2024-12-28
```

> **Each row will become one record in Pinecone. But what exactly goes into each record?**

## Chapter 5 — The design: one row → one record ★

From Part 0, a Pinecone record has three parts: **id + values (the embedding) + metadata**. For every tab we make
four decisions:

| Decision | Question to ask | Rule |
|---|---|---|
| **1 What to embed** | What will users' questions *look like*? | Embed text that **resembles the question** — descriptions, synonyms, example questions |
| **2 What id** | How will I find this record again to update or delete it? | A **stable name** built from the row's own key — never a random number |
| **3 What metadata** | What will I **filter** on? What must come **back** with a match? | Labels to filter (`sheet`, `table`) + what the LLM needs (the text, the SQL) |
| **4 What to skip** | Does this row or field carry meaning a search can use? | Skip what has no searchable meaning |

### Six walk-throughs — one real row from each tab

Each follows the "What was our GMV last month?" story: the table, the column, how it connects, what GMV means, a
proven query, and the limits.

#### 1 · `nlq_table` — what tables exist

```text
ROW (nlq_table)                                    RECORD
───────────────                                    ──────
table:             ecom.orders                     id:       table::ecom.orders
description:       # Orders                        EMBED:    "# Orders  **TABLE:** ecom.orders …
                   Every order placed on the        ──►        Every order placed on the online store
                   online store between 2023-01-01             between 2023-01-01 and 2024-12-28: who
                   and 2024-12-28: who ordered,                ordered, when, how they paid, where it
                   when, how they paid … (Markdown)            ships … (the whole description)"
grain:             One row per order               metadata: {sheet: nlq_table, table: ecom.orders, text}
row_count:         50000              ✗ not embedded
answers_questions: What is total revenue by month?; …  ✗ not embedded
```

The description is already a full Markdown write-up — grain, key measures, business meaning — so it is embedded as
it is. `row_count` is a number with no searchable meaning. `answers_questions` is kept out on purpose: Part 3 uses
those questions as the **test** — if they were embedded, the test would be an open-book exam.

#### 2 · `nlq_column` — which column holds it

```text
ROW (nlq_column)                                   RECORD
────────────────                                   ──────
table:            ecom.orders                      id:       column::ecom.orders.total_amount
column:           total_amount            ──►      EMBED:    "Column ecom.orders.total_amount (FLOAT).
data_type:        FLOAT                                       Merchandise subtotal = SUM(order_items.
description:      Merchandise subtotal …                      total_price); the amount paid. …
analyst_synonyms: Order Total; Revenue; …                     Also called: Order Total; Order Value;
sample_values:    55.92; 437.88; …   ✗ not embedded           Revenue; Sales; Subtotal."
                                                   metadata: {sheet: nlq_column, table: ecom.orders,
                                                              column: total_amount, data_type: FLOAT, text}
```

The analyst synonyms go **into** the text — users say "revenue", not "total_amount". `sample_values` are numbers:
"55.92" means nothing to a meaning search. `table` and `column` are metadata so a search can be narrowed to one table.

#### 3 · `nlq_join` — how two tables connect

```text
ROW (nlq_join)                                     RECORD
──────────────                                     ──────
left_table:  ecom.orders                           id:       join::ecom.orders->ecom.customers::customer_id
right_table: ecom.customers               ──►      EMBED:    "Join ecom.orders to ecom.customers on
join_on:     customer_id                                      customer_id (many-to-one). Each order
cardinality: many-to-one                                      belongs to one customer; a customer has
description: Each order belongs to one                        many orders (56 customers have none)."
             customer; a customer has many …       metadata: {sheet: nlq_join,
                                                              tables: [ecom.orders, ecom.customers], text}
```

A join row is five short fields; we **write them into one sentence** so it reads like the question it answers
("how do orders connect to customers?"). Both tables go into one metadata **list**, `tables`, so a filter on
either table finds this join.

#### 4 · `glossary` — what a business word means here

```text
ROW (glossary)                                     RECORD
──────────────                                     ──────
term:     Revenue                                  id:       glossary::Revenue
category: metric                        ──►        EMBED:    "Revenue (metric): Merchandise sales:
meaning:  Merchandise sales:                                  SUM(orders.total_amount) — shipping, tax
          SUM(orders.total_amount) — shipping,                and discount NOT included …
          tax and discount NOT included …                     Also called: sales; gross sales; GMV; …
synonyms: sales; gross sales; GMV; turnover …                 Columns: ecom.orders.total_amount; …"
maps_to:  ecom.orders.total_amount;                metadata: {sheet: glossary, term: Revenue,
          ecom.order_items.total_price                        category: metric, maps_to: [2 columns], text}
```

**Why the synonyms are embedded:** the word **GMV** appears in exactly one place in the whole workbook — this
row's synonyms. Embed them, and "What was our GMV…" lands right here. Leave them out, and the model has to guess
that GMV means Revenue. This is how we teach the assistant **company jargon**. `maps_to` is also kept as a metadata
**list**: chapter 9 follows it to fetch the exact column records.

#### 5 · `nlq_examples` — a proven query to copy

```text
ROW (nlq_examples)                                 RECORD
──────────────────                                 ──────
question:    What is total merchandise             id:       nlq_examples::1
             revenue by month?            ──►      EMBED:    "What is total merchandise revenue by
paraphrases: monthly revenue; sales per                       month? Also asked as: monthly revenue;
             month; revenue trend by month                    sales per month; revenue trend by month"
query:       SELECT date_trunc('month', …)         metadata: {sheet: nlq_examples, tables_used: [ecom.orders],
             … ✗ not embedded                                 pattern: aggregation,
tables_used: ecom.orders                                      sql: "SELECT date_trunc('month', …", text}
pattern:     aggregation
verified:    Y    owner, source, … ✗ not embedded
```

The **question and its paraphrases** are embedded — they look like what users ask. The **SQL is not embedded**:
nobody searches with SQL. It rides along in metadata (`sql`) and comes back with the match, ready for the LLM.
Bookkeeping fields (`verified`, `owner`, `source`) help the people maintaining the workbook, not the search.

#### 6 · `capability_card` — what the assistant can't answer

```text
ROW (capability_card)                              RECORD
─────────────────────                              ──────
source:      playbook                              id:       capability_card::2
description: WHAT IT CANNOT ANSWER —      ──►      EMBED:    "WHAT IT CANNOT ANSWER — orders after
             orders after 2024-12-28                          2024-12-28 (shipments run to 2024-12-30 …);
             (shipments run to 2024-12-30 …);                 web traffic, sessions or conversion …"
             web traffic, sessions …               metadata: {sheet: capability_card, source: playbook, text}
```

The description is embedded as it is, so "website visitors per day" can find "no visit data". There are only 3 of
these records and every question needs the limits — so chapter 9 also **always includes** them, search or not.

### The design for all six tabs

| Tab | 1 Embed (what questions look like) | 2 Id | 3 Metadata (filter · hand back) | 4 Skip |
|---|---|---|---|---|
| `nlq_table` | the table's description | `table::ecom.orders` | sheet, table · text | — |
| `nlq_column` | column name, type, description, synonyms, allowed values | `column::ecom.orders.total_amount` | sheet, table, column, data_type · text | **8 rows** with `embed_exempt = Y`: map coordinates (`lat`, `lon`) |
| `nlq_join` | "Join A to B on key (cardinality). description" | `join::ecom.orders->ecom.customers::customer_id` | sheet, tables (both) · text | — |
| `glossary` | term, category, meaning, **synonyms**, columns | `glossary::Revenue` | sheet, term, category, maps_to · text | — |
| `nlq_examples` | the **question** + its paraphrases — **not the SQL** | `nlq_examples::12` | sheet, tables_used, pattern · text, **sql** | — |
| `capability_card` | the description | `capability_card::2` | sheet, source · text | — |

Two decisions to discuss in class:

- **Why is the SQL in metadata, not embedded?** Users ask questions, not SQL. We embed the *question* so a user's
  question finds it; the SQL rides along in metadata and comes back with the match — ready to hand to the LLM.
- **Why a stable id?** The workbook changes. When a row is updated, an upsert with the **same id** replaces the old
  record. A random id would leave a duplicate behind (chapter 10).

Two Pinecone rules the code respects: metadata values must be text, numbers, true/false or lists of text — never
empty (`None`); and each record's metadata must stay under 40 KB (our largest is about 2.6 KB).

In [13]:
def as_list(value):
    """Convert a semicolon-separated cell into a Python list."""

    # If the cell is empty, return an empty list.
    if not value:
        return []

    # Split the cell at each semicolon.
    # strip() removes spaces around each item.
    # The if condition skips empty items.
    # Example: "orders; customers" becomes ["orders", "customers"].
    return [
        part.strip()
        for part in str(value).split(";")
        if part.strip()
    ]

In [14]:
def clean(metadata):
    """Remove empty values from a metadata dictionary."""

    # Create a new dictionary to hold only useful metadata values.
    cleaned = {}

    # Check each metadata key and its value.
    for key, value in metadata.items():

        # Keep the value if it is not None, an empty string, or an empty list.
        if value is not None and value != "" and value != []:
            cleaned[key] = value

    # Return metadata without empty values.
    return cleaned

In [15]:
def table_record(row):
    """Build a record from one row in the nlq_table worksheet."""

    # Use the table description as the text that will later be embedded.
    text = row["description"]

    # Create an ID that identifies this table record.
    record_id = f"table::{row['table']}"

    # Store information about where this record came from.
    metadata = {
        "sheet": "nlq_table",
        "table": row["table"],
        "text": text,
    }

    # Return the record ID, the text to embed, and cleaned metadata.
    return record_id, text, clean(metadata)


In [16]:
def column_record(row):
    """Build a record from one row in the nlq_column worksheet."""

    # Start the searchable text with the column name, type, and description.
    text_parts = [
        f"Column {row['table']}.{row['column']} "
        f"({row['data_type']}). {row['description']}"
    ]

    # Add alternate names when the spreadsheet provides them.
    if row.get("analyst_synonyms"):
        text_parts.append(f"Also called: {row['analyst_synonyms']}.")

    # Add possible column values when they are provided.
    if row.get("enum_values"):
        text_parts.append(f"Values: {row['enum_values']}.")

    # Join the text parts into one string for embedding.
    text = " ".join(text_parts)

    # Create an ID using the table and column names.
    record_id = f"column::{row['table']}.{row['column']}"

    # Save details that may be useful when searching or displaying results.
    metadata = {
        "sheet": "nlq_column",
        "table": row["table"],
        "column": row["column"],
        "data_type": row["data_type"],
        "text": text,
    }

    # Return the record ID, text, and cleaned metadata.
    return record_id, text, clean(metadata)

In [17]:
def join_record(row):
    """Build a record from one row in the nlq_join worksheet."""

    # Describe the two tables, the join columns, and the relationship.
    text = (
        f"Join {row['left_table']} to {row['right_table']} "
        f"on {row['join_on']} ({row['cardinality']}). "
        f"{row['description']}"
    )

    # Create an ID based on the tables and join condition.
    record_id = (
        f"join::{row['left_table']}->{row['right_table']}::{row['join_on']}"
    )

    # Store the worksheet name and the tables used by this join.
    metadata = {
        "sheet": "nlq_join",
        "tables": [row["left_table"], row["right_table"]],
        "text": text,
    }

    # Return the record ID, text, and cleaned metadata.
    return record_id, text, clean(metadata)

In [18]:
def glossary_record(row):
    """Build a record from one row in the glossary worksheet."""

    # Start with the glossary term and its meaning.
    text_parts = [f"{row['term']}: {row['meaning']}"]

    # Add synonyms when the term has alternate names.
    if row.get("synonyms"):
        text_parts.append(f"Also called: {row['synonyms']}.")

    # Add mapped columns when the spreadsheet provides them.
    if row.get("maps_to"):
        text_parts.append(f"Related columns: {row['maps_to']}.")

    # Combine all the text into one string for embedding.
    text = " ".join(text_parts)

    # Create an ID using the glossary term.
    record_id = f"glossary::{row['term']}"

    # Save the glossary details as metadata.
    metadata = {
        "sheet": "glossary",
        "term": row["term"],
        "category": row["category"],

        # Convert a semicolon-separated cell into a list.
        "maps_to": as_list(row.get("maps_to")),

        "text": text,
    }

    # Return the record ID, text, and cleaned metadata.
    return record_id, text, clean(metadata)

In [19]:
def example_record(row, number):
    """Build a record from one row in the nlq_examples worksheet."""

    # Use the natural-language question as the main searchable text.
    text_parts = [row["question"]]

    # Add alternate versions of the question, if they are available.
    if row.get("paraphrases"):
        text_parts.append(f"Also asked as: {row['paraphrases']}")

    # Combine the question and its alternate phrasings.
    # The SQL query is stored as metadata; it is not included in the text.
    text = " ".join(text_parts)

    # Use the row number to make this example's ID unique.
    record_id = f"nlq_examples::{number}"

    # Save details about the example for later use.
    metadata = {
        "sheet": "nlq_examples",

        # Convert the table names into a list.
        "tables_used": as_list(row.get("tables_used")),

        "pattern": row.get("pattern"),
        "sql": row.get("query"),
        "text": text,
    }

    # Return the record ID, text, and cleaned metadata.
    return record_id, text, clean(metadata)

In [20]:
def capability_record(row, number):
    """Build a record from one row in the capability_card worksheet."""

    # Use the capability description as the searchable text.
    text = row["description"]

    # Use the row number to make this capability record's ID unique.
    record_id = f"capability_card::{number}"

    # Save the worksheet name and source information.
    metadata = {
        "sheet": "capability_card",
        "source": row.get("source"),
        "text": text,
    }

    # Return the record ID, text, and cleaned metadata.
    return record_id, text, clean(metadata)

In [21]:
records = []

# Go through the worksheet tabs one at a time.
for sheet_name in KB_TABS:

    # Read all non-empty rows from the current worksheet.
    rows = read_sheet(sheet_name)

    # Process each row. The row number starts at 1.
    for number, row in enumerate(rows, start=1):

        # Call the function that matches the current worksheet.
        if sheet_name == "nlq_table":
            record_id, text, metadata = table_record(row)

        elif sheet_name == "nlq_column":
            record_id, text, metadata = column_record(row)

        elif sheet_name == "nlq_join":
            record_id, text, metadata = join_record(row)

        elif sheet_name == "glossary":
            record_id, text, metadata = glossary_record(row)

        elif sheet_name == "nlq_examples":
            record_id, text, metadata = example_record(row, number)

        elif sheet_name == "capability_card":
            record_id, text, metadata = capability_record(row, number)

        # Save the completed record for later embedding or upload.
        records.append((record_id, text, metadata))


# Close the workbook because all worksheet data has been read.
workbook.close()

# Show how many records were built.
print(f"Built {len(records)} records.")

Built 345 records.


## Chapter 6 — Build every record

Run each tab's builder over every row. Three rows-level rules decide what is loaded:

| Rule | Column in the workbook | Today |
|---|---|---|
| Skip rows with no searchable meaning | `embed_exempt = Y` | 8 coordinate columns |
| Skip rows switched off | `active = False` | none |
| Skip rows marked for removal | `action = DELETE` | none — every row is an INSERT |

All three are checked even when they match nothing today: the workbook will change (chapter 10).

In [23]:

workbook = openpyxl.load_workbook(
    "./docs/kb_feed_ecom.xlsx",
    read_only=True,
)

def is_yes(value):
    """Return True when a cell contains y, yes, or true."""
    return str(value).strip().lower() in ("y", "yes", "true")


def should_skip(row):
    """Skip rows that should not be included in the knowledge base."""
    if is_yes(row.get("embed_exempt")):
        return True

    if str(row.get("active", "")).strip().lower() == "false":
        return True

    if str(row.get("action", "")).strip().upper() == "DELETE":
        return True

    return False


records = []

for tab in KB_TABS:
    kept = 0
    skipped = 0

    # Read each row from the current worksheet.
    for number, row in enumerate(read_sheet(tab), start=1):

        # Ignore rows marked as exempt, inactive, or for deletion.
        if should_skip(row):
            skipped += 1
            continue

        # Use the builder for this worksheet to make the record.
        if tab == "nlq_table":
            record_id, text, metadata = table_record(row)
        elif tab == "nlq_column":
            record_id, text, metadata = column_record(row)
        elif tab == "nlq_join":
            record_id, text, metadata = join_record(row)
        elif tab == "glossary":
            record_id, text, metadata = glossary_record(row)
        elif tab == "nlq_examples":
            record_id, text, metadata = example_record(row, number)
        elif tab == "capability_card":
            record_id, text, metadata = capability_record(row, number)

        # Save the record and count it as kept.
        records.append({
            "id": record_id,
            "text": text,
            "metadata": metadata,
        })
        kept += 1

    print(f"{tab}: kept {kept}, skipped {skipped}")


# Make sure no two records have the same ID.
record_ids = [record["id"] for record in records]
assert len(record_ids) == len(set(record_ids)), "Two records share an ID"

print(f"Total: {len(records)} records")

nlq_table: kept 14, skipped 0
nlq_column: kept 177, skipped 8
nlq_join: kept 16, skipped 0
glossary: kept 66, skipped 0
nlq_examples: kept 61, skipped 0
capability_card: kept 3, skipped 0
Total: 337 records


## Chapter 7 — Load into Pinecone

Embed the text and upsert — **100 records per call**, far faster than 337 separate calls.

```text
records[0:100]    ──embed──► 100 × 1536 numbers ──upsert──► namespace "ecom-kb"
records[100:200]  ──embed──► …
records[200:300]  ──embed──► …
records[300:337]  ──embed──► …
```

Same index as Part 0, new namespace: `help-articles` (Part 0) and `ecom-kb` (this notebook) never mix.

In [24]:
BATCH_SIZE = 100

# Process up to 100 records at a time.
for start in range(0, len(records), BATCH_SIZE):
    batch = records[start : start + BATCH_SIZE]

    # Embed the text from each record in this batch.
    texts = []
    for record in batch:
        texts.append(record["text"])

    embeddings = embed(texts)

    # Pair each record with its embedding.
    vectors = []
    for record, embedding in zip(batch, embeddings):
        vector = {
            "id": record["id"],
            "values": embedding,
            "metadata": record["metadata"],
        }
        vectors.append(vector)

    # Upload this batch to Pinecone.
    response = index.upsert(
        vectors=vectors,
        namespace=NAMESPACE,
    )

    print(f"Uploaded {response.upserted_count} records")

print("Finished uploading records.")

Uploaded 100 records
Uploaded 100 records
Uploaded 100 records
Uploaded 37 records
Finished uploading records.


## Chapter 8 — Retrieve: the agent's three lookups

First, the manager's question against the **whole** knowledge base:

In [25]:
def lookup(question, sheets=None, top_k=5):
    # Convert the question into an embedding vector.
    question_vector = embed([question])[0]

    # Search all worksheets unless a list of sheet names is provided.
    search_filter = None
    if sheets:
        search_filter = {"sheet": {"$in": sheets}}

    # Ask Pinecone for the closest matches.
    results = index.query(
        vector=question_vector,
        top_k=top_k,
        namespace=NAMESPACE,
        include_metadata=True,
        filter=search_filter,
    )

    # Return the matches so another function can display or use them.
    return results.matches


def show(title, matches):
    print(title)

    # Pinecone returns matches in similarity order, closest first.
    for match in matches:
        metadata = match.metadata or {}
        text = metadata.get("text", "No text available")

        # Show a short summary of each match.
        print("Score:", round(match.score, 3))
        print("ID:", match.id)
        print("Text:", text[:60])
        print()

    # Pinecone's similarity score is not a confidence percentage.


question = "What was our GMV last month?"

matches = lookup(question, top_k=8)
show(f"WHOLE KNOWLEDGE BASE — {question}", matches)

WHOLE KNOWLEDGE BASE — What was our GMV last month?
Score: 0.505
ID: nlq_examples::8
Text: How did monthly revenue change from the previous month? Also

Score: 0.457
ID: nlq_examples::1
Text: What is total merchandise revenue by month? Also asked as: m

Score: 0.423
ID: glossary::Revenue
Text: Revenue: Merchandise sales: SUM(orders.total_amount) — shipp

Score: 0.377
ID: nlq_examples::43
Text: How many orders are cancelled or still processing, by month?

Score: 0.371
ID: nlq_examples::23
Text: How many new customer accounts were created each month? Also

Score: 0.365
ID: nlq_examples::14
Text: What gross profit did each category earn on its sales? Also 

Score: 0.357
ID: nlq_examples::48
Text: What is the inventory value at cost by category? Also asked 

Score: 0.34
ID: nlq_examples::9
Text: How many gift orders were placed and what is their average v



Look for `glossary::Revenue` — reachable only through the word GMV in its synonyms. Notice what else comes back:
a search over everything returns a **mix** — some useful facts, some noise (columns with "last" in their name, for
"last month"?), all competing for the same top-K slots.

So the agent doesn't search once. It runs **three lookups**, each limited to the tabs it needs — the filter from
Part 0, applied to the `sheet` metadata:

| Agent tool (Version 3) | Searches tabs | Brings back |
|---|---|---|
| `lookup_schema` | `nlq_table`, `nlq_column`, `nlq_join` | where the data lives |
| `lookup_examples` | `nlq_examples` | proven questions + their SQL |
| `lookup_common` | `glossary`, `capability_card` | what words mean here, and the limits |

In [26]:
# Search the table, column, and join information.
schema_matches = lookup(
    question,
    sheets=["nlq_table", "nlq_column", "nlq_join"],
    top_k=3,
)
show("Schema results", schema_matches)

# Search example questions.
example_matches = lookup(
    question,
    sheets=["nlq_examples"],
    top_k=3,
)
show("Example results", example_matches)

# Search glossary terms and capability information.
common_matches = lookup(
    question,
    sheets=["glossary", "capability_card"],
    top_k=3,
)
show("Glossary and capability results", common_matches)

# Keep the results together so later code can use them.
retrieved = {
    "lookup_schema": schema_matches,
    "lookup_examples": example_matches,
    "lookup_common": common_matches,
}

Schema results
Score: 0.298
ID: table::ecom.inventory
Text: # Inventory
**TABLE:** ecom.inventory
**BUSINESS DOMAIN:** E

Score: 0.288
ID: column::ecom.customers.lifetime_revenue
Text: Column ecom.customers.lifetime_revenue (FLOAT). Total mercha

Score: 0.284
ID: column::ecom.products.margin_pct
Text: Column ecom.products.margin_pct (FLOAT). Gross margin percen

Example results
Score: 0.506
ID: nlq_examples::8
Text: How did monthly revenue change from the previous month? Also

Score: 0.457
ID: nlq_examples::1
Text: What is total merchandise revenue by month? Also asked as: m

Score: 0.378
ID: nlq_examples::43
Text: How many orders are cancelled or still processing, by month?

Glossary and capability results
Score: 0.423
ID: glossary::Revenue
Text: Revenue: Merchandise sales: SUM(orders.total_amount) — shipp

Score: 0.324
ID: glossary::VIP
Text: VIP: Customer segment label VIP — 8,083 of 10,000 customers 

Score: 0.324
ID: capability_card::2
Text: WHAT IT CANNOT ANSWER — orders after 20

You get pieces, not the path. The agent solves this differently: it reads the `nlq_join` records and **follows the
connections in code**, from the table holding revenue to the table holding suppliers. Vector search finds the
pieces; code connects them.

---

## The story in one breath

An LLM can write SQL but doesn't know **this** database → the facts it needs are in the workbook, but all of it is
~42,000 tokens per question → so we store each row as a **record** (embed what questions look like, a stable id,
metadata to filter, link and hand back) → load 337 records into Pinecone → for each question the agent runs **three
lookups**, follows the links, adds the limits → and sends the LLM a small **context pack** instead of everything → stable ids keep the index in sync →
and code, not vector search, connects tables.

## Quick knowledge check

1. Name two facts the LLM needs for "What was our GMV last month?" that it can't know by itself.
2. Why not paste the whole workbook into every prompt?
3. What are the four design decisions for each tab?
4. Why are the glossary synonyms embedded?
5. Why is an example's SQL stored in metadata instead of being embedded?
6. Why must ids be stable?
7. Which rows are skipped, and why?
8. What are the agent's three lookups, and which tabs does each search?
9. What is the context pack, and what are the three ways a fact gets into it?
10. Why can't vector search alone answer "revenue by supplier"?

The answers are in the next cell — try first.

## Answers

| # | Answer |
|---|---|
| 1 | Any two of: the table is `ecom.orders`; the column is `total_amount`; GMV here means merchandise-only Revenue; orders end 2024-12-28 |
| 2 | ~42,000 tokens per question: costly, slow, noisy — and a real company's knowledge wouldn't fit at all |
| 3 | What to embed · what id · what metadata · what to skip |
| 4 | Users use those words. "GMV" appears only in the Revenue synonyms; embedding it lets a GMV question find Revenue |
| 5 | Users ask questions, not SQL — so we embed the question; the SQL comes back with the match, ready for the LLM |
| 6 | So an UPDATE replaces the same record (no duplicate) and a DELETE knows exactly which record to remove |
| 7 | `embed_exempt = Y` (8 map coordinates — no searchable meaning), `active = False` and `action = DELETE` (none today) |
| 8 | `lookup_schema` (tables, columns, joins) · `lookup_examples` (examples) · `lookup_common` (glossary, capability card) |
| 9 | What the LLM receives instead of the whole workbook: retrieved facts (glossary, schema, closest example + SQL), columns followed through `maps_to`, and the capability card, always included |
| 10 | It returns records similar in meaning; it doesn't work out which joins connect the tables, in which order. Code follows the `nlq_join` records for that |

**Try it yourself**

1. Ask `"website visitors per day"` with `lookup_common`. Which capability-card record comes first — and what
   should the assistant answer?
2. Remove `Also called: {row['synonyms']}` from `glossary_record`, rebuild and reload. Does the GMV question still
   find `glossary::Revenue`?
3. Change `top_k` in the three lookups to 1 and to 10. How do the context pack's size and quality change?

**Next — Part 3:** measure how often retrieval finds the right records, with ground truth.